# PipelineWatch-NG — Module 1: Data Ingestion
**Project:** Satellite-based crude oil theft & pipeline monitoring — Niger Delta, Nigeria  
**Module goal:** Install dependencies, authenticate APIs, define AOI, download Sentinel-1 SAR tiles and NASA FIRMS fire data for the Trans Niger Pipeline corridor.

> ☁️ **Cloud note:** Run this notebook in Google Colab. All data is stored in Google Drive. No local installation needed.


## 1.1 — Mount Google Drive & install dependencies

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/PipelineWatch_NG'
os.makedirs(f'{PROJECT_ROOT}/data/raw/sentinel1', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/raw/firms',     exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/raw/sentinel2', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/raw/ais',       exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data/processed',     exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/config',             exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs',            exist_ok=True)
print('✅ Drive mounted. Project folders ready.')


In [ ]:
# Install all required packages (run once per Colab session)
!pip install sentinelsat geopandas rioxarray rasterio folium tqdm pyyaml -q
print('✅ Packages installed.')


## 1.2 — Configure API credentials & AOI

In [ ]:
# ─── FILL IN YOUR CREDENTIALS HERE ───────────────────────────────────────────
# Copernicus Data Space: https://dataspace.copernicus.eu  (free registration)
CDSE_USER     = "your_copernicus_email@example.com"
CDSE_PASSWORD = "your_copernicus_password"

# NASA FIRMS MAP KEY: https://firms.modaps.eosdis.nasa.gov/api/area/  (free, instant)
FIRMS_MAP_KEY = "your_firms_map_key"

# Date range of interest
DATE_START = "2024-01-01"
DATE_END   = "2024-03-31"

# Niger Delta AOI — Trans Niger Pipeline + Nembe Creek Trunk Line corridor
# (west, south, east, north)
AOI_BOUNDS = (5.8, 4.2, 7.2, 5.2)
AOI_WKT = "POLYGON((5.8 4.2, 7.2 4.2, 7.2 5.2, 5.8 5.2, 5.8 4.2))"

print('✅ Config loaded.')
print(f'   AOI: lon {AOI_BOUNDS[0]}–{AOI_BOUNDS[2]}  lat {AOI_BOUNDS[1]}–{AOI_BOUNDS[3]}')
print(f'   Period: {DATE_START} → {DATE_END}')


In [ ]:
# Save AOI as GeoJSON for reuse in later modules
import json

aoi_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {
            "name": "Niger Delta TNP corridor",
            "corridor": "Trans Niger Pipeline + Nembe Creek"
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [[[5.8,4.2],[7.2,4.2],[7.2,5.2],[5.8,5.2],[5.8,4.2]]]
        }
    }]
}

aoi_path = f'{PROJECT_ROOT}/config/aoi_niger_delta.geojson'
with open(aoi_path, 'w') as f:
    json.dump(aoi_geojson, f, indent=2)
print(f'✅ AOI saved → {aoi_path}')


## 1.3 — Visualise the AOI on an interactive map

In [ ]:
import folium
import geopandas as gpd

m = folium.Map(location=[4.7, 6.5], zoom_start=8, tiles='CartoDB positron')

# AOI bounding box
folium.Rectangle(
    bounds=[[AOI_BOUNDS[1], AOI_BOUNDS[0]], [AOI_BOUNDS[3], AOI_BOUNDS[2]]],
    color='#e63946', fill=True, fill_opacity=0.08, weight=2,
    tooltip='Niger Delta — TNP Corridor AOI'
).add_to(m)

# Known bunkering hotspot markers
hotspots = [
    (4.95, 6.85, 'Bodo Creek — major spill history'),
    (4.45, 6.10, 'Nembe Creek — high theft activity'),
    (4.62, 6.43, 'Cawthorne Channel'),
    (5.02, 6.70, 'Ogoniland'),
]
for lat, lon, label in hotspots:
    folium.CircleMarker(
        location=[lat, lon], radius=7,
        color='#ff6b35', fill=True, fill_opacity=0.8,
        tooltip=label
    ).add_to(m)

folium.LayerControl().add_to(m)
map_path = f'{PROJECT_ROOT}/outputs/module1_aoi_map.html'
m.save(map_path)
print(f'✅ AOI map saved → {map_path}')
m


## 1.4 — Sentinel-1 SAR: search available scenes

In [ ]:
from sentinelsat import SentinelAPI, geojson_to_wkt, read_geojson
import pandas as pd

# Authenticate against Copernicus Data Space Ecosystem (CDSE)
api = SentinelAPI(
    user=CDSE_USER,
    password=CDSE_PASSWORD,
    api_url='https://catalogue.dataspace.copernicus.eu/odata/v1',
    show_progressbars=True
)
print('✅ Authenticated with CDSE.')


In [ ]:
# Search Sentinel-1 GRD IW VV scenes over Niger Delta
products = api.query(
    AOI_WKT,
    date=(DATE_START, DATE_END),
    platformname='Sentinel-1',
    producttype='GRD',
    sensoroperationalmode='IW',
    polarisationmode='VV VH',
)

df_s1 = api.to_dataframe(products)
df_s1 = df_s1.sort_values('beginposition', ascending=False)

print(f'✅ Found {len(df_s1)} Sentinel-1 GRD scenes')
print()
print(df_s1[['beginposition','size','orbitdirection','relativeorbitnumber']].head(10).to_string())


In [ ]:
# Save scene catalogue (so you don't need to re-query)
catalogue_path = f'{PROJECT_ROOT}/data/raw/sentinel1/scene_catalogue.csv'
df_s1.to_csv(catalogue_path)
print(f'✅ Scene catalogue saved → {catalogue_path}')
print(f'   Total scenes available: {len(df_s1)}')
print(f'   Date range: {df_s1["beginposition"].min()} → {df_s1["beginposition"].max()}')


## 1.5 — Sentinel-1 SAR: download scenes
> ⚠️ Each GRD tile is ~800MB. Start with `MAX_DOWNLOAD = 2` to verify auth, then increase.

In [ ]:
MAX_DOWNLOAD = 2   # ← increase to download more scenes

subset = df_s1.head(MAX_DOWNLOAD)
s1_dir = f'{PROJECT_ROOT}/data/raw/sentinel1/'

print(f'Downloading {MAX_DOWNLOAD} Sentinel-1 scenes...')
print('Sizes:', subset['size'].tolist())
print()

api.download_all(subset.index.tolist(), directory_path=s1_dir)

import glob
downloaded = glob.glob(f'{s1_dir}/*.zip') + glob.glob(f'{s1_dir}/*.SAFE')
print(f'\n✅ Download complete. Files in sentinel1/:')
for f in downloaded:
    print(f'   {os.path.basename(f)}')


## 1.6 — NASA FIRMS: download VIIRS active fire data

In [ ]:
import requests
import io

def fetch_firms(map_key, source, bounds, date_str, days=1):
    """
    Fetch VIIRS fire detections via FIRMS Area API.
    bounds: (west, south, east, north)
    Returns a DataFrame.
    """
    w, s, e, n = bounds
    url = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
           f"{map_key}/{source}/{w},{s},{e},{n}/{date_str}/{days}")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    if len(resp.text.strip()) < 50:   # empty response
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(resp.text))

print('FIRMS client ready.')
print('Sources available: VIIRS_SNPP_NRT (375m), MODIS_NRT (1km)')


In [ ]:
from datetime import datetime, timedelta

FIRMS_SOURCE = 'VIIRS_SNPP_NRT'   # 375m near-real-time

start = datetime.strptime(DATE_START, '%Y-%m-%d')
end   = datetime.strptime(DATE_END,   '%Y-%m-%d')

all_fire = []
current  = start
firms_dir = f'{PROJECT_ROOT}/data/raw/firms/'

print(f'Fetching FIRMS data {DATE_START} → {DATE_END}...')
while current <= end:
    d = current.strftime('%Y-%m-%d')
    try:
        df_day = fetch_firms(FIRMS_MAP_KEY, FIRMS_SOURCE, AOI_BOUNDS, d, days=1)
        if not df_day.empty:
            all_fire.append(df_day)
            print(f'  {d}: {len(df_day):4d} fire pixels')
        else:
            print(f'  {d}: no detections')
    except Exception as ex:
        print(f'  {d}: ERROR — {ex}')
    current += timedelta(days=1)

if all_fire:
    df_firms = pd.concat(all_fire, ignore_index=True)
    firms_path = f'{firms_dir}/firms_viirs_{DATE_START}_{DATE_END}.csv'
    df_firms.to_csv(firms_path, index=False)
    print(f'\n✅ FIRMS data saved → {firms_path}')
    print(f'   Total fire pixels : {len(df_firms)}')
    print(f'   High-FRP (>50 MW) : {(df_firms["frp"] > 50).sum()} — potential illegal refineries')
else:
    print('No FIRMS data retrieved. Check MAP_KEY and date range.')
    df_firms = pd.DataFrame()


## 1.7 — Quick-look: FIRMS fire map

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point, box

if not df_firms.empty:
    gdf_fire = gpd.GeoDataFrame(
        df_firms,
        geometry=gpd.points_from_xy(df_firms.longitude, df_firms.latitude),
        crs='EPSG:4326'
    )
    aoi_poly = gpd.GeoDataFrame(geometry=[box(*AOI_BOUNDS)], crs='EPSG:4326')

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('Module 1 — FIRMS VIIRS Fire Detections: Niger Delta Corridor', fontsize=14)

    # Left: all detections
    ax = axes[0]
    aoi_poly.plot(ax=ax, facecolor='#f8f9fa', edgecolor='#e63946', linewidth=2)
    gdf_fire.plot(ax=ax, column='frp', cmap='hot_r', markersize=8,
                  legend=True, legend_kwds={'label':'FRP (MW)', 'shrink':0.6}, alpha=0.7)
    ax.set_title(f'All fire detections (n={len(gdf_fire)})')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

    # Right: high-FRP only (>50 MW — likely illegal refineries)
    ax2 = axes[1]
    high = gdf_fire[gdf_fire['frp'] > 50]
    aoi_poly.plot(ax=ax2, facecolor='#f8f9fa', edgecolor='#e63946', linewidth=2)
    high.plot(ax=ax2, column='frp', cmap='inferno', markersize=high['frp']/3,
              legend=True, legend_kwds={'label':'FRP >50 MW', 'shrink':0.6}, alpha=0.85)
    ax2.set_title(f'High-FRP only — potential illegal refineries (n={len(high)})')
    ax2.set_xlabel('Longitude')

    plt.tight_layout()
    fig_path = f'{PROJECT_ROOT}/outputs/module1_firms_quicklook.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Figure saved → {fig_path}')
else:
    print('No FIRMS data to plot.')


## 1.8 — Module 1 summary & handoff to Module 2

In [ ]:
print('='*55)
print('MODULE 1 — DATA INGESTION COMPLETE')
print('='*55)

import glob
s1_files = glob.glob(f'{PROJECT_ROOT}/data/raw/sentinel1/*.zip') +            glob.glob(f'{PROJECT_ROOT}/data/raw/sentinel1/*.SAFE')
print(f'  Sentinel-1 SAR tiles  : {len(s1_files)} files')

firms_files = glob.glob(f'{PROJECT_ROOT}/data/raw/firms/*.csv')
total_fire  = len(df_firms) if 'df_firms' in dir() and not df_firms.empty else 0
print(f'  FIRMS fire CSV files  : {len(firms_files)} files')
print(f'  Total fire detections : {total_fire}')
print()
print('Outputs:')
print(f'  {PROJECT_ROOT}/outputs/module1_aoi_map.html')
print(f'  {PROJECT_ROOT}/outputs/module1_firms_quicklook.png')
print()
print('Next → Module 2: SAR preprocessing + oil spill dark-spot detection')
